# 02b - Membersihkan Data
Materi Modul 2: menangani nilai yang hilang dan pencilan (termasuk dengan residual).
Data sengaja dikotori (missing value) lalu dibersihkan, supaya tiap langkah terlihat.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

rng = np.random.default_rng(42)
df = pd.read_csv('../datasets/wine.csv')

## 1. Simulasi missing value

In [ ]:
mask = rng.random(df.shape) < 0.03
mask[:, -1] = False   # kolom target jangan dikotori
df_kotor = df.mask(mask)
df_kotor.isna().sum()

## 2. Imputasi (median)

In [ ]:
imp = SimpleImputer(strategy='median')
X = pd.DataFrame(imp.fit_transform(df_kotor.drop(columns='target')), columns=df.columns[:-1])
print('Missing setelah imputasi:', X.isna().sum().sum())

## 3. Deteksi outlier (IQR)

In [ ]:
Q1, Q3 = X.quantile(0.25), X.quantile(0.75)
IQR = Q3 - Q1
outlier = ((X < Q1 - 1.5*IQR) | (X > Q3 + 1.5*IQR)).sum()
outlier[outlier > 0].sort_values(ascending=False)

## 4. Scaling

In [ ]:
X_std = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
X_mm  = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

fig, ax = plt.subplots(1, 3, figsize=(15,4))
for a, d, t in zip(ax, [X, X_std, X_mm], ['Asli','StandardScaler','MinMaxScaler']):
    d[['alcohol','proline']].boxplot(ax=a); a.set_title(t)
plt.show()

## 3b. Pencilan berbasis residual
Fit regresi linear sederhana, lalu tandai titik dengan standardized residual > 3.

In [ ]:
from sklearn.linear_model import LinearRegression

xr, yr = X[['flavanoids']], X['total_phenols']
lr = LinearRegression().fit(xr, yr)
res = yr - lr.predict(xr)
z = (res - res.mean()) / res.std()
pencilan = z.abs() > 3
print('Jumlah pencilan (|z residual| > 3):', pencilan.sum())
plt.scatter(xr, yr, c=pencilan, cmap='coolwarm')
plt.xlabel('flavanoids'); plt.ylabel('total_phenols'); plt.title('Pencilan berdasarkan residual'); plt.show()

In [ ]:
# Opsi penanganan: hapus pencilan
X_bersih = X.loc[~pencilan].reset_index(drop=True)
print(X.shape, '->', X_bersih.shape)

## Catatan
- (kenapa pilih median, kapan pakai StandardScaler vs MinMax, dll.)